# Optimization vs Sampling

Chapter 18: Langevin Dynamics and Sampling · *Mathematics for AI and Machine Learning* by Fuheng Wu

Gradient ascent vs Langevin dynamics on a 2D Gaussian mixture.

Companion code for the [interactive demo](https://math4ai.org/companion/chapter-18/opt-vs-sampling/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import torch

torch.manual_seed(42)

# The demo's target: p(x) ∝ exp(-|x - m1|^2 / 1.6) + exp(-|x - m2|^2 / 1.6), start (0, -3.5)
m = torch.tensor([[-2.0, -1.0], [2.0, 1.5]], dtype=torch.float64)
var = 0.8
eta, opt_steps, lang_steps = 0.05, 50, 800
start = torch.tensor([0.0, -3.5], dtype=torch.float64)


def score(x):
    """grad log p for the two-bump mixture (works on a batch of points)."""
    w = torch.softmax(-((x[:, None, :] - m) ** 2).sum(-1) / (2 * var), dim=1)
    return (w[:, :, None] * (-(x[:, None, :] - m) / var)).sum(1)


# ── Gradient ascent on log p: deterministic, stops at the nearest mode ──────
x = start[None].clone()
for _ in range(opt_steps):
    x = x + eta * score(x)
print("gradient ascent ends at", [round(v, 3) for v in x[0].tolist()], "(a mode)")

# ── Langevin: the same drift plus sqrt(2 eta) noise keeps exploring ─────────
chains = start.repeat(2000, 1)
print("\nLangevin, 2000 chains from the same start:")
for k in range(1, 4001):
    chains = chains + eta * score(chains) + (2 * eta) ** 0.5 * torch.randn_like(chains)
    if k in (lang_steps, 4000):
        near = ((chains[:, None, :] - m) ** 2).sum(-1).argmin(1)
        print(f"  after {k:4d} steps: near mode 1 {(near == 0).double().mean().item():.3f}, "
              f"near mode 2 {(near == 1).double().mean().item():.3f}  (target 0.5 / 0.5), "
              f"mean {[round(v, 2) for v in chains.mean(0).tolist()]} (target {m.mean(0).tolist()})")
# The start is closer to mode 1, so it takes a while for chains to spread out.
# Optimization returns one point; sampling returns the whole distribution.

gradient ascent ends at [-1.921, -1.099] (a mode)

Langevin, 2000 chains from the same start:
  after  800 steps: near mode 1 0.593, near mode 2 0.407  (target 0.5 / 0.5), mean [-0.39, -0.01] (target [0.0, 0.25])


  after 4000 steps: near mode 1 0.479, near mode 2 0.520  (target 0.5 / 0.5), mean [0.04, 0.32] (target [0.0, 0.25])
